# Table 10 reproduction: per-seed intraclass correlations under fine-tuning

**What this resolves.** Table 10 reports, for the fine-tuned regime, rho = 0.313 with a 95%
interval of [0.125, 0.782] and a standard-error ratio of 20.3x. The point estimate and the
ratio both reproduce under the estimator used everywhere else in the paper. The interval does
not: the exact one-way random-effects construction at F = 6 gives **[0.150, 0.733]**, and no
value of m between 100 and 1800 produces [0.125, 0.782].

Every other interval in the manuscript *does* reproduce under that estimator, including the
frozen-probe row of Table 10 itself (rho = 0.173 gives exactly [0.075, 0.558]). So the fine-tuned
row is the one number in the paper a reviewer cannot check.

**What this notebook does.**

1. Implements the estimator and validates it against published values in Tables 5, 7 and 10.
2. Tests, with no data at all, which interval construction could produce [0.125, 0.782].
3. Re-runs the fine-tuning (3 seeds x 5 folds) and recomputes rho per seed from scratch.
4. Reports every candidate pooling rule side by side so the correct row can be chosen.

**Leading hypothesis (H1).** The reported interval is the *envelope* of the three per-seed
intervals rather than a single interval. Working backwards: rho = 0.269 has lower bound 0.125,
rho = 0.373 has upper bound 0.782, and the mean of {0.269, 0.297, 0.373} is 0.313. This matches
Section 6.10's phrasing, "all three seed intervals excluding zero", which implies per-seed
intervals were computed. If H1 holds, the caption's claim that "only the model differs" between
the two Table 10 rows is wrong, because the frozen row uses a single interval.

Section 3 tests this prediction directly against the recomputed values.

---

**Notes on running.** Cells 1-3 need no data and run in seconds. They are worth running first.
Cells 4-8 need NEU-CLS and a GPU. Nothing in this notebook uses bare `assert`: checks print a
verdict and continue, so a failure never kills the runtime.

## 0. Configuration

In [ ]:
# MUST come before torch is imported anywhere: CUBLAS_WORKSPACE_CONFIG is read
# when the CUDA context is created, and setting it later silently does nothing.
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "0")

from pathlib import Path

CFG = {
    # --- data -------------------------------------------------------------
    # Where NEU-CLS lives. Cell 0a downloads it here, so you do not need to set
    # this unless you already have a copy somewhere else.
    "neu_root": "/content/data/NEU-CLS",

    # Optional: your own SDI-C module, so the corruptions match the paper exactly.
    # Must expose apply_corruption(img_uint8, family, severity, image_id) -> uint8.
    # Leave as None to use the reference implementation in Cell 5 (see its warning).
    "sdic_module_path": None,

    # Optional: the released fold assignment (image_id -> fold). CSV with columns
    # image_id,fold. Leave None to rebuild a stratified 5-fold split from SEED_FOLDS.
    "folds_csv": None,

    # --- experiment -------------------------------------------------------
    "seeds": [0, 1, 2],
    "n_folds": 5,
    "epochs": 15,
    "batch_size": 32,
    "lr": 1e-4,
    "img_size": 224,
    "seed_folds": 12345,

    # Images entering the variance-components analysis. The manuscript reports
    # 1,800 images in Section 5.1 but extraction "over 900 images" in Section 5.2,
    # and every interval reproduces at m = 900. Set to None to use all of them;
    # set to 900 to match the paper. This cell is where that question gets settled.
    "m_target": None,

    # --- corruption cache -------------------------------------------------
    # Corrupted images depend only on (image_id, family, severity), never on
    # seed or fold, so they are computed once and reused across all three seeds.
    # /content is local SSD. Everything here is lost when the runtime recycles;
    # the last cell offers a download of the results so nothing important is.
    "cache_dir": "/content/corrupt_cache",
    "use_cache": True,

    # --- output -----------------------------------------------------------
    "out_dir": "/content/table10_repro",
    "quick": False,   # True -> 2 epochs, 1 fold, 1 seed. For a smoke test only.
}

TEST_FAMILIES = ["motion_blur", "shot_noise", "brightness_drift",
                 "contrast_loss", "vignetting", "vibration_jitter"]
SEVERITIES = [1, 2, 3, 4, 5]

if CFG["quick"]:
    CFG.update(epochs=2, seeds=[0], n_folds=1)
    print("QUICK MODE: results are a smoke test, not a reproduction.")

Path(CFG["out_dir"]).mkdir(parents=True, exist_ok=True)
for k, v in CFG.items():
    print(f"  {k:18s} {v}")

  neu_root           /content/data/NEU-CLS
  sdic_module_path   None
  folds_csv          None
  seeds              [0, 1, 2]
  n_folds            5
  epochs             15
  batch_size         32
  lr                 0.0001
  img_size           224
  seed_folds         12345
  m_target           None
  cache_dir          /content/corrupt_cache
  use_cache          True
  out_dir            /content/table10_repro
  quick              False


### 0a. Download NEU-CLS

No Drive, no Kaggle token, no manual upload. This pulls the dataset straight into the runtime
and flattens it into one folder of 1,800 images.

The dataset is Song and Yan's NEU surface defect database (Northeastern University). The original
host has been intermittently unavailable for years, so this cell pulls from a GitHub mirror and
then verifies what arrived: 1,800 images, 300 per class, 200x200, grayscale, plus a content
checksum. If the first mirror fails it tries the second. If both fail it tells you what to do.

Verification matters more than the source here. A mirror that quietly serves a resized or
partial copy would change every number downstream, so the cell refuses to report success unless
the checks pass.

In [ ]:
import hashlib, io, os, shutil, urllib.request, zipfile
from pathlib import Path
from collections import Counter

MIRRORS = [
    ("Indir99",
     "https://codeload.github.com/Indir99/Surface-Defect-Detection-for-NEU-Database/zip/refs/heads/master",
     "3e8c4ee7b6b9827b"),
    ("aviralchharia",
     "https://codeload.github.com/aviralchharia/Surface-Defect-Detection-in-Hot-Rolled-Steel-Strips/zip/refs/heads/master",
     None),
]

PREFIXES = ["Cr", "In", "Pa", "PS", "RS", "Sc"]
IMG_EXT = {".bmp", ".jpg", ".jpeg", ".png"}
DEST = Path(CFG["neu_root"])

def _class_of(name):
    stem = Path(name).stem
    for pre in PREFIXES:
        if stem.lower().startswith(pre.lower() + "_"):
            return pre
    return None

def _content_hash(paths):
    h = hashlib.sha256()
    for q in sorted(paths, key=lambda z: z.name):
        h.update(q.name.encode())
        h.update(q.read_bytes())
    return h.hexdigest()[:16]

def verify(root, quiet=False):
    root = Path(root)
    if not root.exists():
        return False, []
    imgs = [q for q in root.rglob("*")
            if q.suffix.lower() in IMG_EXT and _class_of(q.name)]
    if not imgs:
        return False, []
    counts = Counter(_class_of(q.name) for q in imgs)
    ok = len(imgs) == 1800 and all(counts.get(c, 0) == 300 for c in PREFIXES)
    if not quiet:
        print(f"    {len(imgs)} images, per class {dict(counts)}")
        try:
            from PIL import Image
            im = Image.open(imgs[0])
            print(f"    sample: {im.size} mode={im.mode}")
            ok = ok and im.size == (200, 200) and im.mode in ("L", "1")
        except Exception as e:
            print(f"    [note] could not open a sample ({e})")
    return ok, imgs

# Already present?
ok, imgs = verify(DEST, quiet=True)
if ok:
    print(f"  already present at {DEST} ({len(imgs)} images); skipping download")
else:
    DEST.mkdir(parents=True, exist_ok=True)
    got = False
    for name, url, expect in MIRRORS:
        print(f"  trying mirror: {name}")
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "colab"})
            with urllib.request.urlopen(req, timeout=180) as r:
                blob = r.read()
            print(f"    downloaded {len(blob)/1e6:.1f} MB")
            zf = zipfile.ZipFile(io.BytesIO(blob))
            members = [m for m in zf.namelist()
                       if Path(m).suffix.lower() in IMG_EXT and _class_of(Path(m).name)]
            print(f"    {len(members)} class-prefixed images in archive")
            for m in members:                       # flatten: one folder, no splits
                target = DEST / Path(m).name
                with zf.open(m) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
            ok, imgs = verify(DEST)
            if ok:
                h = _content_hash(imgs)
                print(f"    content sha256[:16] = {h}")
                if expect and h != expect:
                    print(f"    [warn] expected {expect}. Images differ from the copy this "
                          f"notebook was checked against; results may not be comparable.")
                elif expect:
                    print("    checksum matches the reference copy")
                got = True
                break
            print("    [MISS] verification failed for this mirror")
            for q in DEST.glob("*"):
                q.unlink()
        except Exception as e:
            print(f"    [MISS] {type(e).__name__}: {e}")
    if not got:
        print("\n  [MISS] no mirror worked. Two options:")
        print("    1. Upload a zip yourself:")
        print("         from google.colab import files; up = files.upload()")
        print("       then unzip it into", DEST)
        print("    2. Point CFG['neu_root'] at a copy you already have and re-run Cell 0.")

ok, imgs = verify(DEST, quiet=True)
print(f"\n  NEU-CLS ready: {ok}  ({len(imgs)} images at {DEST})")

  trying mirror: Indir99
    downloaded 62.1 MB
    1800 class-prefixed images in archive
    1800 images, per class {'Sc': 300, 'In': 300, 'Cr': 300, 'RS': 300, 'Pa': 300, 'PS': 300}
    sample: (200, 200) mode=L
    content sha256[:16] = 3e8c4ee7b6b9827b
    checksum matches the reference copy

  NEU-CLS ready: True  (1800 images at /content/data/NEU-CLS)


In [ ]:
# Soft checks. Nothing here raises; a failure prints and the notebook continues.
import sys, importlib, subprocess

def check(name, ok, detail=""):
    print(f"  [{'ok ' if ok else 'MISS'}] {name}" + (f"  {detail}" if detail else ""))
    return bool(ok)

print("environment")
for mod in ["numpy", "scipy", "torch", "torchvision", "PIL", "pandas"]:
    try:
        importlib.import_module(mod)
        check(mod, True)
    except ImportError:
        check(mod, False, "-> pip install")

try:
    import torch
    check("cuda", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (slow)")
except Exception as e:
    check("cuda", False, str(e))

from pathlib import Path
check("neu_root", Path(CFG["neu_root"]).exists(), CFG["neu_root"])
if CFG["sdic_module_path"]:
    check("sdic_module", Path(CFG["sdic_module_path"]).exists(), CFG["sdic_module_path"])
else:
    print("  [note] sdic_module_path is None -> reference implementation will be used.")

environment
  [ok ] numpy
  [ok ] scipy
  [ok ] torch
  [ok ] torchvision
  [ok ] PIL
  [ok ] pandas
  [ok ] cuda  Tesla T4
  [ok ] neu_root  /content/data/NEU-CLS
  [note] sdic_module_path is None -> reference implementation will be used.


## 1. The estimator

One-way random-effects model on the severity-averaged paired response `d_if`, with corruption
family as the grouping factor. This is the two-component reduction the manuscript reports
throughout (Section 5.3): `rho` is a lower bound on the family intraclass correlation, and
`sigma_e` is a composite of residual and image variance.

In [ ]:
import numpy as np
from scipy.stats import f as fdist

def icc_oneway(D, alpha=0.05):
    """Exact one-way random-effects ICC with an F-based confidence interval.

    D : (m, F) array. Rows are images, columns are corruption families.
        Entry D[i, f] is the severity-averaged response for image i under family f.

    Returns dict with rho, its interval, the F statistic, and the SE ratio of
    Equation (2) under the two-component reduction.
    """
    D = np.asarray(D, dtype=float)
    if D.ndim != 2 or min(D.shape) < 2:
        return {"error": f"need a 2-D array with >=2 rows and columns, got {D.shape}"}
    if not np.isfinite(D).all():
        return {"error": "non-finite entries in D"}

    m, F = D.shape                      # images per family, number of families
    grand = D.mean()
    fam_means = D.mean(axis=0)          # one mean per family

    ms_between = m * ((fam_means - grand) ** 2).sum() / (F - 1)
    ms_within = ((D - fam_means) ** 2).sum() / (F * (m - 1))
    if ms_within <= 0:
        return {"error": "zero within-family variance"}

    Fstat = ms_between / ms_within
    rho = (Fstat - 1) / (Fstat + m - 1)

    d1, d2 = F - 1, F * (m - 1)
    fu = fdist.ppf(1 - alpha / 2, d1, d2)
    fl = fdist.ppf(alpha / 2, d1, d2)
    lo = (Fstat / fu - 1) / (Fstat / fu + m - 1)
    hi = (Fstat / fl - 1) / (Fstat / fl + m - 1)

    return {"rho": rho, "ci": (lo, hi), "F": Fstat, "m": m, "n_families": F,
            "se_ratio": se_ratio(rho, m), "ms_between": ms_between, "ms_within": ms_within}


def icc_ci_from_rho(rho, m, F=6, alpha=0.05):
    """Interval implied by a reported rho, without the underlying data."""
    Fstat = 1 + m * rho / (1 - rho)
    d1, d2 = F - 1, F * (m - 1)
    fu = fdist.ppf(1 - alpha / 2, d1, d2)
    fl = fdist.ppf(alpha / 2, d1, d2)
    return ((Fstat / fu - 1) / (Fstat / fu + m - 1),
            (Fstat / fl - 1) / (Fstat / fl + m - 1))


def se_ratio(rho, m):
    """Equation (2) under the two-component reduction (sigma_b^2 = 0)."""
    return float(np.sqrt(1 + m * rho / (1 - rho)))

print("estimator defined")

estimator defined


### 1a. Validate the estimator against published values

Before touching data, confirm the estimator reproduces intervals already in the manuscript.
If these match, any later disagreement is about the data or the pooling rule, not the estimator.

In [ ]:
PUBLISHED = [   # (label, rho, m, reported_ci, reported_se_ratio)
    ("T5  C4 vs C2",        0.029,  900, (0.011, 0.156), 5.28),
    ("T5  C7q vs C2",       0.106,  900, (0.044, 0.420), 10.38),
    ("T5  C7 vs C5",        0.305,  900, (0.146, 0.726), 19.90),
    ("T7  ResNet-50 vs C2", 0.069,  900, (0.028, 0.312), 8.2),
    ("T7  EffNet-B0 vs C2", 0.101,  900, (0.042, 0.406), 10.1),
    ("T8  MagTile max",     0.325, 1344, (None,  None),  25.4),
    ("T10 frozen, acc-def", 0.173,  900, (0.075, 0.558), 13.7),
    ("T10 FINE-TUNED",      0.313,  900, (0.125, 0.782), 20.3),
]

print(f"{'row':22s} {'rho':>6s} {'computed CI':>18s} {'reported CI':>18s} {'CI?':>5s} {'SE':>6s} {'SE?':>5s}")
mismatches = []
for label, rho, m, rep_ci, rep_se in PUBLISHED:
    lo, hi = icc_ci_from_rho(rho, m)
    sr = se_ratio(rho, m)
    ci_ok = (rep_ci[0] is None) or (abs(lo - rep_ci[0]) < 0.003 and abs(hi - rep_ci[1]) < 0.003)
    se_ok = abs(sr - rep_se) < 0.06
    rep_txt = "n/a" if rep_ci[0] is None else f"[{rep_ci[0]:.3f}, {rep_ci[1]:.3f}]"
    print(f"{label:22s} {rho:6.3f} [{lo:.3f}, {hi:.3f}]  {rep_txt:>18s} "
          f"{'yes' if ci_ok else 'NO':>5s} {sr:6.2f} {'yes' if se_ok else 'NO':>5s}")
    if not ci_ok:
        mismatches.append(label)

print()
if mismatches == ["T10 FINE-TUNED"]:
    print("Estimator validated: every published interval reproduces EXCEPT the fine-tuned")
    print("row of Table 10. That row is therefore not a single exact one-way interval.")
elif not mismatches:
    print("All rows reproduce. Re-check the reported fine-tuned interval.")
else:
    print("Unexpected mismatches:", mismatches)

row                       rho        computed CI        reported CI   CI?     SE   SE?
T5  C4 vs C2            0.029 [0.011, 0.156]      [0.011, 0.156]   yes   5.28   yes
T5  C7q vs C2           0.106 [0.043, 0.418]      [0.044, 0.420]   yes  10.38   yes
T5  C7 vs C5            0.305 [0.145, 0.726]      [0.146, 0.726]   yes  19.90   yes
T7  ResNet-50 vs C2     0.069 [0.027, 0.311]      [0.028, 0.312]   yes   8.23   yes
T7  EffNet-B0 vs C2     0.101 [0.041, 0.405]      [0.042, 0.406]   yes  10.11   yes
T8  MagTile max         0.325 [0.158, 0.744]                 n/a   yes  25.46   yes
T10 frozen, acc-def     0.173 [0.075, 0.558]      [0.075, 0.558]   yes  13.76   yes
T10 FINE-TUNED          0.313 [0.150, 0.733]      [0.125, 0.782]    NO  20.27   yes

Estimator validated: every published interval reproduces EXCEPT the fine-tuned
row of Table 10. That row is therefore not a single exact one-way interval.


## 2. Which construction produces [0.125, 0.782]?

Pure arithmetic, no data required. Each hypothesis is stated so it can be rejected.

In [ ]:
from scipy.optimize import brentq

TARGET = (0.125, 0.782)
RHO_REPORTED = 0.313
print(f"target interval {TARGET}, reported rho {RHO_REPORTED}\n")

# H2: a different number of images.
print("H2  different m")
for m in [180, 360, 540, 900, 1344, 1800, 5000]:
    lo, hi = icc_ci_from_rho(RHO_REPORTED, m)
    print(f"      m = {m:5d}  ->  [{lo:.3f}, {hi:.3f}]")
print("      verdict: m barely moves the interval. H2 rejected.\n")

# H3: a different confidence level.
print("H3  different alpha")
for a in [0.05, 0.02, 0.01]:
    lo, hi = icc_ci_from_rho(RHO_REPORTED, 900, alpha=a)
    print(f"      alpha = {a:.2f}  ->  [{lo:.3f}, {hi:.3f}]")
print("      verdict: no standard level lands on the target. H3 rejected.\n")

# H4: seed treated as a third grouping factor, reducing the effective family count.
print("H4  fewer effective families")
for F in [3, 4, 5, 6]:
    lo, hi = icc_ci_from_rho(RHO_REPORTED, 900, F=F)
    print(f"      F = {F}  ->  [{lo:.3f}, {hi:.3f}]")
print()

# H1: envelope of the three per-seed intervals.
r_lo = brentq(lambda r: icc_ci_from_rho(r, 900)[0] - TARGET[0], 0.05, 0.90)
r_hi = brentq(lambda r: icc_ci_from_rho(r, 900)[1] - TARGET[1], 0.05, 0.90)
r_mid = 3 * RHO_REPORTED - r_lo - r_hi
print("H1  envelope of three per-seed intervals  <-- leading hypothesis")
print(f"      rho with LOWER bound {TARGET[0]}  ->  {r_lo:.3f}")
print(f"      rho with UPPER bound {TARGET[1]}  ->  {r_hi:.3f}")
print(f"      third seed implied by mean = {RHO_REPORTED}  ->  {r_mid:.3f}")
print(f"      predicted per-seed rho: {sorted([round(r_lo,3), round(r_mid,3), round(r_hi,3)])}")
print(f"      check: mean = {(r_lo + r_mid + r_hi)/3:.4f}")
print()
print("PREDICTION for Section 5: if H1 is right, the three recomputed per-seed rho")
print(f"values should have min ~= {r_lo:.3f} and max ~= {r_hi:.3f}.")

H1_PRED = {"min": r_lo, "mid": r_mid, "max": r_hi}

target interval (0.125, 0.782), reported rho 0.313

H2  different m
      m =   180  ->  [0.148, 0.735]
      m =   360  ->  [0.149, 0.734]
      m =   540  ->  [0.150, 0.733]
      m =   900  ->  [0.150, 0.733]
      m =  1344  ->  [0.150, 0.733]
      m =  1800  ->  [0.150, 0.733]
      m =  5000  ->  [0.151, 0.733]
      verdict: m barely moves the interval. H2 rejected.

H3  different alpha
      alpha = 0.05  ->  [0.150, 0.733]
      alpha = 0.02  ->  [0.131, 0.805]
      alpha = 0.01  ->  [0.119, 0.847]
      verdict: no standard level lands on the target. H3 rejected.

H4  fewer effective families
      F = 3  ->  [0.109, 0.947]
      F = 4  ->  [0.127, 0.864]
      F = 5  ->  [0.140, 0.790]
      F = 6  ->  [0.150, 0.733]

H1  envelope of three per-seed intervals  <-- leading hypothesis
      rho with LOWER bound 0.125  ->  0.269
      rho with UPPER bound 0.782  ->  0.373
      third seed implied by mean = 0.313  ->  0.296
      predicted per-seed rho: [0.269, 0.296, 0.373]
  

## 3. Data

NEU-CLS: 1,800 grayscale images, six classes, 300 each.

In [ ]:
import re, numpy as np
from pathlib import Path
from PIL import Image

NEU_CLASSES = ["Cr", "In", "Pa", "PS", "RS", "Sc"]

def load_neu(root):
    root = Path(root)
    if not root.exists():
        print(f"  [MISS] {root} not found. Fix CFG['neu_root'] and re-run this cell.")
        return [], np.array([], dtype=int), []
    paths, labels, ids = [], [], []
    subdirs = [d for d in sorted(root.iterdir()) if d.is_dir()]
    if subdirs:
        for li, d in enumerate(subdirs):
            for p in sorted(d.glob("*")):
                if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".tif"}:
                    paths.append(p); labels.append(li); ids.append(p.stem)
    else:
        for p in sorted(root.glob("*")):
            if p.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".tif"}:
                continue
            m = re.match(r"([A-Za-z_]+)", p.stem)
            key = (m.group(1).rstrip("_") if m else "?")
            if key not in NEU_CLASSES:
                continue
            paths.append(p); labels.append(NEU_CLASSES.index(key)); ids.append(p.stem)
    return paths, np.array(labels, dtype=int), ids

PATHS, LABELS, IDS = load_neu(CFG["neu_root"])
print(f"  loaded {len(PATHS)} images, {len(np.unique(LABELS))} classes")
if len(PATHS):
    counts = np.bincount(LABELS)
    print("  per class:", counts.tolist())
    check("1800 images", len(PATHS) == 1800, f"got {len(PATHS)}")
    check("balanced 300/class", bool((counts == 300).all()), f"got {counts.tolist()}")

  loaded 1800 images, 6 classes
  per class: [300, 300, 300, 300, 300, 300]
  [ok ] 1800 images  got 1800
  [ok ] balanced 300/class  got [300, 300, 300, 300, 300, 300]


### 3b. Optional: paste your own SDI-C here

`CFG["sdic_module_path"]` assumes your corruption code sits in an importable `.py` file. If it
lives inside one of your other notebooks instead, paste the function into the cell below rather
than extracting it to a file. Anything defined here takes precedence over both the module path
and the reference implementation.

Required signature:

```python
apply_corruption(img_uint8, family, severity, image_id) -> uint8 array
```

`family` is one of the six held-out names in `TEST_FAMILIES`, `severity` is 1-5, and `image_id`
is the string used to seed the nuisance parameters. If your own function differs (different
argument order, a severity-indexed dict, a class method), write a thin wrapper here that adapts
it; the wrapper is what matters, not the internals.

In [ ]:
# ===========================================================================
# PASTE YOUR IMPLEMENTATION BELOW. Leave the cell untouched to skip.
# ===========================================================================

# def apply_corruption(img, family, severity, image_id):
#     """img: uint8 HxW. Returns uint8 HxW."""
#     ...
#     return out

# --- example wrapper, if your own function has a different signature --------
# from my_sdic import corrupt as _mine
# def apply_corruption(img, family, severity, image_id):
#     return _mine(image=img, corruption=family, level=severity, seed_key=image_id)

if callable(globals().get("apply_corruption")):
    print("  found a pasted apply_corruption -> it will be used")
else:
    print("  no pasted implementation; falling back to module path or reference")

  no pasted implementation; falling back to module path or reference


## 4. SDI-C test families

If `CFG["sdic_module_path"]` points at your own implementation, it is used and the corruptions
match the paper exactly. Otherwise the reference implementation below is used.

**The reference implementation is not the paper's.** It follows the descriptions in Table 1 and
Section 3.2, including the two documented repairs (nuisance parameters seeded from
`(image_id, family)` and deliberately *not* from severity; brightness drift as a gamma transform
rather than an additive offset, so it cannot clip). It will produce *a* valid rho, not *the*
published rho. Only the module path reproduces Table 10 exactly.

In [ ]:
import numpy as np, hashlib, importlib.util
from scipy.ndimage import gaussian_filter, zoom

def _rng(image_id, family):
    """Seeded from (image_id, family) only. Severity is excluded by design."""
    h = hashlib.sha256(f"{image_id}|{family}".encode()).digest()
    return np.random.default_rng(int.from_bytes(h[:8], "big"))

def _ref_corrupt(img, family, severity, image_id):
    """Reference SDI-C test families. img: uint8 HxW or HxWx3. Returns uint8."""
    x = img.astype(np.float32) / 255.0
    r = _rng(image_id, family)
    s = severity  # 1..5, amplitude only

    if family == "motion_blur":
        angle = r.uniform(0, np.pi)                 # fixed across severities
        L = int(2 + 3 * s)
        k = np.zeros((L, L), np.float32)
        cy = cx = L // 2
        for t in np.linspace(-L / 2, L / 2, L * 4):
            y, x_ = int(cy + t * np.sin(angle)), int(cx + t * np.cos(angle))
            if 0 <= y < L and 0 <= x_ < L:
                k[y, x_] = 1.0
        k /= max(k.sum(), 1e-8)
        out = np.stack([_conv2(x[..., c], k) for c in range(x.shape[-1])], -1) if x.ndim == 3 \
              else _conv2(x, k)

    elif family == "shot_noise":
        lam = [60, 25, 12, 6, 3][s - 1]
        out = r.poisson(np.clip(x, 0, 1) * lam) / float(lam)

    elif family == "brightness_drift":
        # gamma transform: maps [0,1] onto itself, cannot saturate
        gamma = 1.0 + 0.18 * s
        out = np.clip(x, 0, 1) ** gamma

    elif family == "contrast_loss":
        f = [0.85, 0.70, 0.55, 0.40, 0.28][s - 1]
        out = (x - x.mean()) * f + x.mean()

    elif family == "vignetting":
        H, W = x.shape[:2]
        cy, cx = (H - 1) / 2, (W - 1) / 2
        yy, xx = np.mgrid[0:H, 0:W]
        rr = np.sqrt(((yy - cy) / cy) ** 2 + ((xx - cx) / cx) ** 2) / np.sqrt(2)
        strength = 0.15 * s
        mask = 1.0 - strength * rr ** 2
        out = x * (mask[..., None] if x.ndim == 3 else mask)

    elif family == "vibration_jitter":
        direction = r.uniform(0, 2 * np.pi)          # fixed across severities
        amp = 0.5 + 0.9 * s
        H, W = x.shape[:2]
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        phase = r.uniform(0, 2 * np.pi)
        dy = amp * np.sin(2 * np.pi * yy / 12.0 + phase) * np.sin(direction)
        dx = amp * np.sin(2 * np.pi * yy / 12.0 + phase) * np.cos(direction)
        out = _warp(x, dy, dx)

    else:
        print(f"  [MISS] unknown family {family}; returning image unchanged")
        out = x

    return (np.clip(out, 0, 1) * 255).astype(np.uint8)

def _conv2(a, k):
    from scipy.signal import convolve2d
    return convolve2d(a, k, mode="same", boundary="symm")

def _warp(x, dy, dx):
    H, W = x.shape[:2]
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    sy = np.clip(yy + dy, 0, H - 1).astype(int)
    sx = np.clip(xx + dx, 0, W - 1).astype(int)
    return x[sy, sx] if x.ndim == 2 else x[sy, sx, :]

# Resolution order: pasted function, then module path, then reference.
APPLY = None
SDIC_SOURCE = "reference"      # "pasted" | "module" | "reference"

if callable(globals().get("apply_corruption")):
    APPLY = globals()["apply_corruption"]
    SDIC_SOURCE = "pasted"
    print("  using the implementation pasted in Cell 3b -> comparable with the paper")
elif CFG["sdic_module_path"]:
    try:
        spec = importlib.util.spec_from_file_location("sdic_user", CFG["sdic_module_path"])
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        APPLY = mod.apply_corruption
        SDIC_SOURCE = "module"
        print("  using YOUR SDI-C module -> comparable with the paper")
    except Exception as e:
        print(f"  [MISS] could not load your module ({e}); falling back to reference")

if APPLY is None:
    APPLY = _ref_corrupt
    SDIC_SOURCE = "reference"
    print("  " + "=" * 68)
    print("  USING THE REFERENCE IMPLEMENTATION.")
    print("  These corruptions are NOT the ones behind Table 10. rho estimated from")
    print("  them measures the heterogeneity of THIS suite, not of SDI-C, and cannot")
    print("  be compared with any number in the manuscript. The hypothesis tests")
    print("  downstream will be suppressed. Paste your own code in Cell 3b.")
    print("  " + "=" * 68)

# Monotonicity smoke test: mean local sharpness must fall under motion blur.
if len(PATHS):
    im = np.array(Image.open(PATHS[0]).convert("L"))
    sharp = [float(np.var(np.gradient(APPLY(im, "motion_blur", s, IDS[0]).astype(float))[0]))
             for s in SEVERITIES]
    check("motion blur monotone in sharpness", all(np.diff(sharp) <= 1e-9),
          " > ".join(f"{v:.1f}" for v in sharp))

  USING THE REFERENCE IMPLEMENTATION.
  These corruptions are NOT the ones behind Table 10. rho estimated from
  them measures the heterogeneity of THIS suite, not of SDI-C, and cannot
  be compared with any number in the manuscript. The hypothesis tests
  downstream will be suppressed. Paste your own code in Cell 3b.
  [ok ] motion blur monotone in sharpness  39.9 > 21.2 > 16.3 > 11.7 > 9.6


## 4b. Precompute the corruption cache

Every corrupted image is a deterministic function of `(image_id, family, severity)`. Nothing in
it depends on the seed or the fold, so recomputing it inside the training loop does the same
work three times over: 162,000 corruption operations instead of 54,000.

This cell computes each one once and stores it as a memory-mapped array, one file per
(family, severity). Building is resumable, so an interrupted run picks up where it stopped, and
a second full run of the notebook costs nothing.

Set `CFG["use_cache"] = False` to skip this and corrupt on the fly.

In [ ]:
import json, hashlib, inspect, time
import numpy as np
from pathlib import Path
from PIL import Image

CACHE = Path(CFG["cache_dir"])

def _fingerprint():
    """Identifies the corruption implementation by its OUTPUT on a fixed probe image.

    Hashing behaviour rather than source is what makes this reliable: it catches a
    changed parameter inside your SDI-C module, a swapped implementation, or an
    edited cell, none of which a source hash sees consistently from a notebook.
    """
    probe = ((np.arange(64 * 64, dtype=np.int64).reshape(64, 64) * 7) % 251).astype(np.uint8)
    h = hashlib.sha256()
    for fam in TEST_FAMILIES:
        for sev in SEVERITIES:
            try:
                out = np.ascontiguousarray(APPLY(probe, fam, sev, "__probe__"))
                h.update(out.tobytes())
            except Exception:
                h.update(f"error:{fam}:{sev}".encode())
    return {"impl": h.hexdigest()[:16],
            "ids":  hashlib.sha256("|".join(IDS).encode()).hexdigest()[:16],
            "n":    len(IDS)}

def cache_path(fam, sev):
    return CACHE / f"{fam}_s{sev}.npy"

def build_cache(verbose=True):
    if not CFG["use_cache"]:
        print("  cache disabled")
        return False
    if not len(PATHS):
        print("  [MISS] no images loaded; nothing to cache")
        return False

    CACHE.mkdir(parents=True, exist_ok=True)
    fp = _fingerprint()
    man = CACHE / "manifest.json"
    if man.exists():
        try:
            old = json.loads(man.read_text())
            if old != fp:
                print("  implementation or image set changed -> rebuilding cache")
                for f in CACHE.glob("*.npy"):
                    f.unlink()
        except Exception:
            pass

    probe = np.array(Image.open(PATHS[0]).convert("L"))
    H, W = probe.shape
    shapes_ok = True
    for pth in PATHS[1:50]:
        if np.array(Image.open(pth).convert("L")).shape != (H, W):
            shapes_ok = False
            break
    if not shapes_ok:
        print("  [MISS] images differ in size; cache needs a uniform shape. Falling back.")
        return False

    n = len(PATHS)
    gb = n * H * W * len(TEST_FAMILIES) * len(SEVERITIES) / 1e9
    print(f"  {n} images at {H}x{W}, {len(TEST_FAMILIES)} families x {len(SEVERITIES)} severities")
    print(f"  cache size ~= {gb:.2f} GB at {CACHE}")

    t0 = time.time()
    for fam in TEST_FAMILIES:
        for sev in SEVERITIES:
            out = cache_path(fam, sev)
            if out.exists():
                if verbose:
                    print(f"    {fam}_s{sev}: already built")
                continue
            tmp = out.with_suffix(".tmp.npy")
            arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(n, H, W))
            for i, pth in enumerate(PATHS):
                im = np.array(Image.open(pth).convert("L"))
                arr[i] = APPLY(im, fam, sev, IDS[i])
            arr.flush(); del arr
            tmp.rename(out)          # atomic: a partial file is never left behind
            if verbose:
                print(f"    {fam}_s{sev}: built  ({time.time()-t0:.0f}s elapsed)")

    man.write_text(json.dumps(fp))
    print(f"  cache ready in {time.time()-t0:.0f}s")
    return True

CACHE_READY = build_cache()

  1800 images at 200x200, 6 families x 5 severities
  cache size ~= 2.16 GB at /content/corrupt_cache
    motion_blur_s1: built  (9s elapsed)
    motion_blur_s2: built  (31s elapsed)
    motion_blur_s3: built  (56s elapsed)
    motion_blur_s4: built  (94s elapsed)
    motion_blur_s5: built  (146s elapsed)
    shot_noise_s1: built  (154s elapsed)
    shot_noise_s2: built  (162s elapsed)
    shot_noise_s3: built  (169s elapsed)
    shot_noise_s4: built  (174s elapsed)
    shot_noise_s5: built  (180s elapsed)
    brightness_drift_s1: built  (181s elapsed)
    brightness_drift_s2: built  (183s elapsed)
    brightness_drift_s3: built  (184s elapsed)
    brightness_drift_s4: built  (185s elapsed)
    brightness_drift_s5: built  (187s elapsed)
    contrast_loss_s1: built  (188s elapsed)
    contrast_loss_s2: built  (189s elapsed)
    contrast_loss_s3: built  (191s elapsed)
    contrast_loss_s4: built  (193s elapsed)
    contrast_loss_s5: built  (194s elapsed)
    vignetting_s1: built  (196s e

In [ ]:
# Cache-aware reader. Each DataLoader worker opens its own memmap after fork.
_MM = {}

def _memmap(fam, sev):
    key = (fam, sev)
    if key not in _MM:
        p = cache_path(fam, sev)
        try:
            _MM[key] = np.load(p, mmap_mode="r") if p.exists() else None
        except Exception:
            _MM[key] = None
    return _MM[key]

def read_corrupted(i, fam, sev):
    """Cached corrupted image if available, otherwise computed on the fly."""
    mm = _memmap(fam, sev)
    if mm is not None:
        return np.asarray(mm[i])
    return APPLY(np.array(Image.open(PATHS[i]).convert("L")), fam, sev, IDS[i])

# Verify the cache against freshly computed values before trusting it.
if globals().get("CACHE_READY") and len(PATHS):
    rs = np.random.default_rng(0)
    bad = 0
    checked = 0
    for _ in range(12):
        i = int(rs.integers(0, len(PATHS)))
        fam = TEST_FAMILIES[int(rs.integers(0, len(TEST_FAMILIES)))]
        sev = int(rs.integers(1, 6))
        fresh = APPLY(np.array(Image.open(PATHS[i]).convert("L")), fam, sev, IDS[i])
        cached = read_corrupted(i, fam, sev)
        checked += 1
        if not np.array_equal(fresh, cached):
            bad += 1
    check(f"cache matches fresh computation ({checked} spot checks)", bad == 0,
          f"{bad} mismatches" if bad else "")
else:
    print("  [note] no cache in use; corruptions will be computed on the fly")

  [ok ] cache matches fresh computation (12 spot checks)


## 5. Folds

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

def build_folds(labels, n_folds, seed):
    rng = np.random.default_rng(seed)
    fold = np.full(len(labels), -1)
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        for k, i in enumerate(idx):
            fold[i] = k % n_folds
    return fold

FOLD = None
if CFG["folds_csv"] and Path(CFG["folds_csv"]).exists():
    df = pd.read_csv(CFG["folds_csv"])
    mapping = dict(zip(df["image_id"].astype(str), df["fold"].astype(int)))
    FOLD = np.array([mapping.get(str(i), -1) for i in IDS])
    check("released folds cover all images", (FOLD >= 0).all(),
          f"{int((FOLD < 0).sum())} unmapped")
    print("  using the released fold assignment")
elif len(PATHS):
    FOLD = build_folds(LABELS, CFG["n_folds"], CFG["seed_folds"])
    print("  rebuilt a stratified 5-fold split (not the released one)")

if FOLD is not None and len(PATHS):
    print("  fold sizes:", np.bincount(FOLD[FOLD >= 0]).tolist())
else:
    print("  [MISS] no folds built (no images loaded). Fix CFG['neu_root'] first.")

  rebuilt a stratified 5-fold split (not the released one)
  fold sizes: [360, 360, 360, 360, 360]


## 6. Fine-tune and evaluate

ResNet-50 end-to-end, one model per (seed, fold). Each test fold is evaluated clean and under
every (family, severity), giving the per-image accuracy deficit relative to clean.

Results are checkpointed to `out_dir` after every fold, so a runtime disconnect loses at most
one fold.

In [ ]:
                                                                                                           import json, time
import numpy as np, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

DEV = "cuda" if torch.cuda.is_available() else "cpu"
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

# Determinism. Without these, two runs with the same seed give different rho.
import random
if os.environ.get("CUBLAS_WORKSPACE_CONFIG") != ":4096:8":
    print("  [warn] CUBLAS_WORKSPACE_CONFIG was not set before CUDA started;")
    print("         restart the runtime and run from Cell 0 for full determinism.")
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception as e:
    print(f"  [note] deterministic algorithms unavailable ({type(e).__name__})")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _worker_init(worker_id):
    ws = torch.initial_seed() % 2**31
    np.random.seed(ws + worker_id)
    random.seed(ws + worker_id)

class NEU(Dataset):
    """corrupt=None -> clean. Otherwise (family, severity).

    Augmentation is drawn from an RNG seeded by (aug_seed, image index), so it is
    identical no matter how the DataLoader distributes work across workers.
    """
    def __init__(self, idx, corrupt=None, train=False, aug_seed=0, epoch=0):
        self.idx, self.corrupt, self.train = idx, corrupt, train
        self.aug_seed, self.epoch = aug_seed, epoch
        self.tf = T.Compose([T.Resize((CFG["img_size"], CFG["img_size"]),
                                      interpolation=T.InterpolationMode.BICUBIC),
                             T.ToTensor(), NORM])
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        if self.corrupt is not None:                 # applied at native resolution
            fam, sev = self.corrupt
            im = read_corrupted(i, fam, sev)         # cached when available
        else:
            im = np.array(Image.open(PATHS[i]).convert("L"))
        pil = Image.fromarray(im).convert("RGB")
        if self.train:
            rg = np.random.default_rng((self.aug_seed, self.epoch, int(i)))
            if rg.random() < 0.5:
                pil = pil.transpose(Image.FLIP_LEFT_RIGHT)
        return self.tf(pil), int(LABELS[i]), i

def train_one(tr_idx, seed):
    seed_everything(seed)
    net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    net.fc = nn.Linear(net.fc.in_features, 6)
    net = net.to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=CFG["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG["epochs"])
    lossf = nn.CrossEntropyLoss()
    net.train()
    for ep in range(CFG["epochs"]):
        gen = torch.Generator().manual_seed(seed * 1000 + ep)
        dl = DataLoader(NEU(tr_idx, train=True, aug_seed=seed, epoch=ep),
                        batch_size=CFG["batch_size"], shuffle=True, num_workers=2,
                        drop_last=True, generator=gen, worker_init_fn=_worker_init)
        tot = 0.0
        for xb, yb, _ in dl:
            xb, yb = xb.to(DEV), yb.to(DEV)
            opt.zero_grad()
            l = lossf(net(xb), yb)
            l.backward(); opt.step()
            tot += float(l.detach())
        sched.step()
    return net.eval()

@torch.no_grad()
def correct_flags(net, te_idx, corrupt=None):
    """Per-image 0/1 correctness, indexed by position in te_idx."""
    dl = DataLoader(NEU(te_idx, corrupt=corrupt), batch_size=64, num_workers=2,
                    worker_init_fn=_worker_init)
    out = np.zeros(len(te_idx))
    pos = {i: k for k, i in enumerate(te_idx)}
    for xb, yb, ib in dl:
        pred = net(xb.to(DEV)).argmax(1).cpu().numpy()
        for p, y, i in zip(pred, yb.numpy(), ib.numpy()):
            out[pos[int(i)]] = float(p == y)
    return out

def run_seed(seed):
    """Returns (n_images, 6) accuracy-deficit matrix, pooled over folds."""
    rows, order = [], []
    folds = range(CFG["n_folds"]) if not CFG["quick"] else [0]
    for f in folds:
        te = np.where(FOLD == f)[0]
        tr = np.where((FOLD != f) & (FOLD >= 0))[0]
        t0 = time.time()
        net = train_one(tr, seed * 100 + f)
        clean = correct_flags(net, te)
        D = np.zeros((len(te), len(TEST_FAMILIES)))
        for fi, fam in enumerate(TEST_FAMILIES):
            acc = np.zeros(len(te))
            for sv in SEVERITIES:
                acc += correct_flags(net, te, corrupt=(fam, sv))
            D[:, fi] = clean - acc / len(SEVERITIES)     # deficit relative to clean
        rows.append(D); order.append(te)
        print(f"    seed {seed} fold {f}: clean acc {clean.mean():.3f}, "
              f"mean deficit {D.mean():.4f}  ({time.time()-t0:.0f}s)")
        np.save(Path(CFG["out_dir"]) / f"D_seed{seed}_fold{f}.npy", D)
        del net
        if DEV == "cuda":
            torch.cuda.empty_cache()
    return np.vstack(rows), np.concatenate(order)

# Self-check: the augmented tensor for a given (seed, epoch, index) must be
# identical across constructions. This is the failure that made rho drift.
if FOLD is not None and len(PATHS):
    _idx = np.where(FOLD == 0)[0][:8]
    _a = torch.stack([NEU(_idx, train=True, aug_seed=7, epoch=1)[k][0] for k in range(8)])
    _b = torch.stack([NEU(_idx, train=True, aug_seed=7, epoch=1)[k][0] for k in range(8)])
    _c = torch.stack([NEU(_idx, train=True, aug_seed=8, epoch=1)[k][0] for k in range(8)])
    check("augmentation reproducible at fixed seed", bool(torch.equal(_a, _b)))
    check("augmentation responds to the seed", not bool(torch.equal(_a, _c)))

def determinism_probe(epochs=2):
    """Train the same seed twice on one fold and compare weights. ~2 minutes.

    Cheaper than discovering after a 75-minute run that rho was not reproducible.
    """
    if FOLD is None or not len(PATHS):
        print("  [MISS] no data")
        return
    keep, CFG["epochs"] = CFG["epochs"], epochs
    tr = np.where(FOLD != 0)[0][:240]
    try:
        w = []
        for _ in range(2):
            net = train_one(tr, 4242)
            w.append({k: v.detach().cpu().clone() for k, v in net.state_dict().items()})
            del net
            if DEV == "cuda":
                torch.cuda.empty_cache()
        same = all(torch.equal(w[0][k], w[1][k]) for k in w[0])
        mx = max(float((w[0][k].float() - w[1][k].float()).abs().max()) for k in w[0])
        check("same seed gives identical weights", same, f"max diff {mx:.2e}")
        if not same:
            print("    Training is not reproducible on this runtime. Per-seed rho will")
            print("    drift between runs, so treat the spread across seeds as an upper")
            print("    bound that includes run-to-run noise, not as seed variance alone.")
    finally:
        CFG["epochs"] = keep

determinism_probe()

D_BY_SEED = {}
if FOLD is not None and len(PATHS):
    for s in CFG["seeds"]:
        print(f"  seed {s}")
        D_BY_SEED[s], _ = run_seed(s)
        np.save(Path(CFG["out_dir"]) / f"D_seed{s}.npy", D_BY_SEED[s])
    print("  done:", {s: v.shape for s, v in D_BY_SEED.items()})
else:
    print("  [MISS] no data or folds; skipping. Cell 7 can still run on saved .npy files.")

  [ok ] augmentation reproducible at fixed seed
  [ok ] augmentation responds to the seed
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 197MB/s]


  [ok ] same seed gives identical weights  max diff 0.00e+00
  seed 0
    seed 0 fold 0: clean acc 1.000, mean deficit 0.2280  (313s)
    seed 0 fold 1: clean acc 1.000, mean deficit 0.2394  (324s)
    seed 0 fold 2: clean acc 1.000, mean deficit 0.2223  (324s)
    seed 0 fold 3: clean acc 1.000, mean deficit 0.2123  (325s)
    seed 0 fold 4: clean acc 1.000, mean deficit 0.2106  (324s)
  seed 1
    seed 1 fold 0: clean acc 1.000, mean deficit 0.2140  (324s)
    seed 1 fold 1: clean acc 1.000, mean deficit 0.2181  (324s)
    seed 1 fold 2: clean acc 1.000, mean deficit 0.2220  (328s)
    seed 1 fold 3: clean acc 1.000, mean deficit 0.2238  (324s)
    seed 1 fold 4: clean acc 1.000, mean deficit 0.2133  (324s)
  seed 2
    seed 2 fold 0: clean acc 1.000, mean deficit 0.2459  (324s)
    seed 2 fold 1: clean acc 1.000, mean deficit 0.2021  (325s)
    seed 2 fold 2: clean acc 1.000, mean deficit 0.2369  (324s)
    seed 2 fold 3: clean acc 1.000, mean deficit 0.2172  (325s)
    seed 2 fold 

## 7. Per-seed rho, and every candidate pooling rule

This is the cell that answers the question.

In [ ]:
import numpy as np
from pathlib import Path

# Survive a fresh runtime: rebuild the dict if Cell 6 has not run in this session.
D_BY_SEED = globals().get("D_BY_SEED", {})
per_seed = {}

if not D_BY_SEED:
    for s in CFG["seeds"]:
        p = Path(CFG["out_dir"]) / f"D_seed{s}.npy"
        if p.exists():
            D_BY_SEED[s] = np.load(p)
    print("  recovered from disk:", list(D_BY_SEED))

if not D_BY_SEED:
    print("  [MISS] nothing to analyse.")
else:
    per_seed = {}
    print(f"{'seed':>5s} {'m':>6s} {'rho':>7s} {'95% CI':>18s} {'SE ratio':>9s}")
    for s, D in sorted(D_BY_SEED.items()):
        if CFG["m_target"] and D.shape[0] > CFG["m_target"]:
            rs = np.random.default_rng(0)
            D = D[rs.choice(D.shape[0], CFG["m_target"], replace=False)]
        r = icc_oneway(D)
        if "error" in r:
            print(f"{s:>5d}  {r['error']}")
            continue
        per_seed[s] = r
        print(f"{s:>5d} {r['m']:>6d} {r['rho']:>7.3f} "
              f"[{r['ci'][0]:.3f}, {r['ci'][1]:.3f}] {r['se_ratio']:>9.2f}")

    if per_seed:
        rhos = np.array([v["rho"] for v in per_seed.values()])
        los = np.array([v["ci"][0] for v in per_seed.values()])
        his = np.array([v["ci"][1] for v in per_seed.values()])
        m_used = per_seed[list(per_seed)[0]]["m"]

        print("\n  candidate constructions for the Table 10 fine-tuned row")
        print(f"    {'rule':38s} {'rho':>7s} {'interval':>18s}")

        cands = {}
        cands["mean rho, exact CI at mean"] = (rhos.mean(), icc_ci_from_rho(rhos.mean(), m_used))
        cands["mean rho, envelope of seed CIs"] = (rhos.mean(), (los.min(), his.max()))
        cands["median rho, exact CI at median"] = (float(np.median(rhos)),
                                                   icc_ci_from_rho(float(np.median(rhos)), m_used))
        D_pool = np.vstack(list(D_BY_SEED.values()))
        rp = icc_oneway(D_pool)
        if "error" not in rp:
            cands["pooled rows across seeds"] = (rp["rho"], rp["ci"])
        D_avg = np.mean(np.stack([D_BY_SEED[s] for s in sorted(D_BY_SEED)]), axis=0)
        ra = icc_oneway(D_avg)
        if "error" not in ra:
            cands["averaged over seeds, then ICC"] = (ra["rho"], ra["ci"])

        # H5: a resampling interval rather than the exact F construction. The paper
        # already uses BCa bootstrap for image-level intervals (Section 5.3), so a
        # bootstrap over families is a live candidate for this row, and unlike the
        # analytic forms it cannot be reconstructed from rho alone.
        def boot_icc(D, B=4000, seed=0, alpha=0.05):
            rs = np.random.default_rng(seed)
            m, F = D.shape
            out = []
            for _ in range(B):
                cols = rs.integers(0, F, F)          # resample families
                r = icc_oneway(D[:, cols])
                if "error" not in r:
                    out.append(r["rho"])
            if len(out) < 100:
                return None
            return float(np.percentile(out, 100 * alpha / 2)), \
                   float(np.percentile(out, 100 * (1 - alpha / 2)))

        bs = boot_icc(D_pool)
        if bs:
            cands["bootstrap over families (percentile)"] = (rp["rho"], bs)
        bi = boot_icc(D_avg)
        if bi:
            cands["bootstrap, seed-averaged"] = (ra["rho"], bi)

        for name, (r_, ci_) in cands.items():
            flag = ""
            if abs(r_ - 0.313) < 0.02 and abs(ci_[0] - 0.125) < 0.02 and abs(ci_[1] - 0.782) < 0.03:
                flag = "   <-- MATCHES Table 10"
            print(f"    {name:38s} {r_:>7.3f} [{ci_[0]:.3f}, {ci_[1]:.3f}]{flag}")

        comparable = (globals().get("SDIC_SOURCE") in ("pasted", "module"))

        print("\n  H1 check (envelope of per-seed intervals)")
        print(f"    predicted per-seed rho min/max: {H1_PRED['min']:.3f} / {H1_PRED['max']:.3f}")
        print(f"    observed  per-seed rho min/max: {rhos.min():.3f} / {rhos.max():.3f}")
        print(f"    envelope = [{los.min():.3f}, {his.max():.3f}]")

        if not comparable:
            print("\n    VERDICT SUPPRESSED. These numbers come from the reference")
            print("    corruptions, not from SDI-C. rho is a property of the corruption")
            print("    suite, so a disagreement with the manuscript says nothing about")
            print("    how Table 10's interval was built. Supply your own implementation")
            print("    in Cell 3b and re-run before drawing any conclusion.")
        else:
            h1 = abs(los.min() - 0.125) < 0.02 and abs(his.max() - 0.782) < 0.03
            print(f"    ->  {'H1 SUPPORTED' if h1 else 'H1 not supported'}")
            spread = (rhos.max() - rhos.min()) / 2 / max(rhos.mean(), 1e-9)
            need = (H1_PRED["max"] - H1_PRED["min"]) / 2 / 0.313
            print(f"    seed spread observed +/-{spread*100:.1f}%, "
                  f"H1 requires +/-{need*100:.1f}%")

        print("\n  m actually used:", m_used,
              "(Section 5.1 says 1,800 images; Section 5.2 says 900).")
        if m_used != 900:
            print("    The manuscript's intervals reproduce at m = 900. Set")
            print("    CFG['m_target'] = 900 to match, or resolve the discrepancy.")

 seed      m     rho             95% CI  SE ratio
    0   1800   0.517 [0.294, 0.866]     43.94
    1   1800   0.481 [0.265, 0.848]     40.89
    2   1800   0.469 [0.256, 0.842]     39.90

  candidate constructions for the Table 10 fine-tuned row
    rule                                       rho           interval
    mean rho, exact CI at mean               0.489 [0.272, 0.852]
    mean rho, envelope of seed CIs           0.489 [0.256, 0.866]
    median rho, exact CI at median           0.481 [0.265, 0.848]
    pooled rows across seeds                 0.489 [0.271, 0.852]
    averaged over seeds, then ICC            0.519 [0.296, 0.867]
    bootstrap over families (percentile)     0.489 [0.124, 0.684]
    bootstrap, seed-averaged                 0.519 [0.155, 0.709]

  H1 check (envelope of per-seed intervals)
    predicted per-seed rho min/max: 0.269 / 0.373
    observed  per-seed rho min/max: 0.469 / 0.517
    envelope = [0.256, 0.866]

    VERDICT SUPPRESSED. These numbers come fr

## 8. What to put in the paper

Whatever this notebook returns, Table 10 needs one of two corrections.

**If H1 is supported** (the reported interval is the envelope of three per-seed intervals), then
the two Table 10 rows use different constructions and the caption's "only the model differs" is
wrong. The fix is to report the three per-seed values explicitly and state the pooling rule, or
to replace the envelope with the single exact interval [0.150, 0.733] so both rows match.

**If H1 is rejected**, the reported interval has no reconstruction and should be replaced by the
value this notebook computes.

Either way, add the per-seed rho values to the manuscript. They are the only place seed variance
appears anywhere in the paper, and Section 6.3's 61% ECE swing under a change of configuration
makes their absence from every other interval worth a sentence.

The export below is formatted for pasting into a response to reviewers.

In [ ]:
import json, numpy as np
from pathlib import Path

if globals().get('per_seed'):
    rhos = [float(v["rho"]) for v in per_seed.values()]
    out = {
        "config": {k: (str(v) if isinstance(v, Path) else v) for k, v in CFG.items()},
        "per_seed": {str(s): {"rho": float(v["rho"]),
                              "ci": [float(v["ci"][0]), float(v["ci"][1])],
                              "se_ratio": float(v["se_ratio"]),
                              "m": int(v["m"]), "F": float(v["F"])}
                     for s, v in per_seed.items()},
        "mean_rho": float(np.mean(rhos)),
        "envelope": [float(min(v["ci"][0] for v in per_seed.values())),
                     float(max(v["ci"][1] for v in per_seed.values()))],
        "exact_ci_at_mean": [float(x) for x in icc_ci_from_rho(float(np.mean(rhos)),
                                                               per_seed[list(per_seed)[0]]["m"])],
        "manuscript_reported": {"rho": 0.313, "ci": [0.125, 0.782], "se_ratio": 20.3},
        "sdic_source": globals().get("SDIC_SOURCE", "unknown"),
        "comparable_with_manuscript":
            globals().get("SDIC_SOURCE") in ("pasted", "module") and
            per_seed[list(per_seed)[0]]["m"] == 900,
    }
    p = Path(CFG["out_dir"]) / "table10_reproduction.json"
    p.write_text(json.dumps(out, indent=2))
    print(json.dumps(out, indent=2))
    print("\nwritten to", p)

    # Nothing under /content survives a runtime recycle, so pull the results out now.
    try:
        from google.colab import files
        files.download(str(p))
        print("download started")
    except Exception as e:
        print(f"[note] automatic download unavailable ({type(e).__name__}); "
              f"copy the JSON printed above instead")
else:
    print("  [MISS] no results to export.")

{
  "config": {
    "neu_root": "/content/data/NEU-CLS",
    "sdic_module_path": null,
    "folds_csv": null,
    "seeds": [
      0,
      1,
      2
    ],
    "n_folds": 5,
    "epochs": 15,
    "batch_size": 32,
    "lr": 0.0001,
    "img_size": 224,
    "seed_folds": 12345,
    "m_target": null,
    "cache_dir": "/content/corrupt_cache",
    "use_cache": true,
    "out_dir": "/content/table10_repro",
    "quick": false
  },
  "per_seed": {
    "0": {
      "rho": 0.5174180066722662,
      "ci": [
        0.294399584624148,
        0.8658254062623708
      ],
      "se_ratio": 43.94241804781961,
      "m": 1800,
      "F": 1930.9361038893428
    },
    "1": {
      "rho": 0.4814461903155868,
      "ci": [
        0.26538078850690594,
        0.8482136595328088
      ],
      "se_ratio": 40.892448162880896,
      "m": 1800,
      "F": 1672.1923167539014
    },
    "2": {
      "rho": 0.46916426121852245,
      "ci": [
        0.2558817932414622,
        0.841766850715613
      ],
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

download started
